# Tangkis: Eksplorasi Data dan Preprocessing

Notebook ini adalah tahap **pengumpulan data** dan **preprocessing** pada Project Awal (RPS MKBL). Hasilnya dipakai untuk melatih model **Jaringan Syaraf Tiruan jenis Multilayer Perceptron** pada tahap berikutnya.

**Isi notebook**
1. Persiapan
2. Memuat dataset
3. Menggabungkan data tambahan dari anggota kelompok
4. Eksplorasi data
5. Preprocessing teks
6. Analisis kosakata
7. Menyimpan hasil
8. Ringkasan untuk slide

**Kelas yang diprediksi**

| ID | Kelas | Contoh |
|---|---|---|
| 0 | normal | Nanti sore jadi kerja kelompok di perpus ya. |
| 1 | penipuan | Selamat! Anda menang undian Rp50 juta. Kirim kode OTP ke nomor ini. |
| 2 | promo | Diskon 50% semua menu hari ini, tunjukkan SMS ini di kasir. |

Cara menjalankan di Google Colab: menu **Runtime > Run all**.

## 1. Persiapan

Library yang dipakai: **pandas** untuk mengolah tabel data, **re** (regular expression) untuk mencari pola teks, dan **matplotlib** untuk grafik (materi MKBL pertemuan 4).

In [ ]:
import json
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.dpi"] = 110

KELAS = ["normal", "penipuan", "promo"]
ID_KELAS = {nama: i for i, nama in enumerate(KELAS)}
WARNA = {"normal": "#2FA36B", "penipuan": "#E2543E", "promo": "#E59A1A"}

FOLDER_HASIL = Path("hasil")
FOLDER_HASIL.mkdir(exist_ok=True)

## 2. Memuat dataset

Dataset utama: **SMS Spam Indonesia** oleh agtbaskara, berisi 1.143 SMS berbahasa Indonesia dengan label `normal`, `penipuan`, dan `promo`.

Label teks diubah menjadi angka (`label_id`) karena keluaran Jaringan Syaraf Tiruan berupa angka. Urutannya sama dengan urutan kelas di aplikasi Android.

Notebook memakai file `dataset_sms_spam_v2.csv` yang diunggah ke Colab (panel **Files** di kiri). Jika file tidak ditemukan, dataset diambil langsung dari internet.

In [ ]:
FILE_DATASET = Path("dataset_sms_spam_v2.csv")
URL_DATASET = (
    "https://gist.githubusercontent.com/agtbaskara/a1a7017027cc1df9d35cf06e1e5575b7"
    "/raw/dataset_sms_spam_v2.csv"
)

sumber_dataset = FILE_DATASET if FILE_DATASET.exists() else URL_DATASET
print("Dataset dimuat dari:", sumber_dataset)

data = pd.read_csv(sumber_dataset).rename(columns={"Teks": "teks"})
data["teks"] = data["teks"].astype(str).str.strip()
data["sumber"] = "SMS Spam Indonesia"

print("Jumlah baris:", len(data))
data.head()

In [ ]:
data.info()
print()
print("Label yang ada:", sorted(data["label"].unique()))

## 3. Menggabungkan data tambahan dari anggota kelompok

Contoh pesan penipuan asli dari HP anggota atau keluarga disimpan di `ml/data/data_tambahan.csv` dengan kolom `teks` dan `label` (`normal`, `penipuan`, atau `promo`).

**Wajib:** nama, nomor HP, dan nomor rekening disamarkan sebelum dimasukkan, misalnya `081234567890` ditulis `08xxxxxxxxxx`.

Notebook mencari file ini di folder Colab terlebih dahulu (upload lewat panel **Files** di kiri), lalu di repository GitHub. Jika belum ada isinya, notebook tetap berjalan dengan dataset utama saja.

In [ ]:
URL_DATA_TAMBAHAN = "https://raw.githubusercontent.com/manrandomside/tangkis/main/ml/data/data_tambahan.csv"


def muat_data_tambahan():
    for lokasi in ["data_tambahan.csv", URL_DATA_TAMBAHAN]:
        try:
            tambahan = pd.read_csv(lokasi)
        except Exception:
            continue
        tambahan = tambahan.dropna(subset=["teks", "label"])
        tambahan["teks"] = tambahan["teks"].astype(str).str.strip()
        tambahan["label"] = tambahan["label"].astype(str).str.strip().str.lower()
        tidak_valid = tambahan[~tambahan["label"].isin(KELAS)]
        if len(tidak_valid):
            print("Baris dengan label tidak dikenal diabaikan:", len(tidak_valid))
        tambahan = tambahan[tambahan["label"].isin(KELAS)]
        tambahan["sumber"] = "Data tambahan kelompok"
        print(f"Data tambahan dimuat dari {lokasi}: {len(tambahan)} baris")
        return tambahan
    print("Data tambahan belum tersedia, memakai dataset utama saja.")
    return pd.DataFrame(columns=["teks", "label", "sumber"])


data_tambahan = muat_data_tambahan()
data = pd.concat([data, data_tambahan], ignore_index=True)

### Membersihkan baris kosong dan duplikat

Pesan yang sama persis cukup disimpan sekali, supaya tidak muncul dua kali di data latih dan data uji.

In [ ]:
jumlah_awal = len(data)
data = data[data["teks"].str.len() > 0]
data = data.drop_duplicates(subset="teks").reset_index(drop=True)
data["label_id"] = data["label"].map(ID_KELAS)

print("Baris awal          :", jumlah_awal)
print("Baris setelah bersih:", len(data))
print("Baris yang dibuang  :", jumlah_awal - len(data))

## 4. Eksplorasi data

### 4.1 Jumlah pesan per kelas

Jumlah data tiap kelas perlu diperiksa karena data yang **tidak seimbang** membuat model cenderung menebak kelas mayoritas. Karena itu evaluasi nanti tidak cukup dengan akurasi saja, tetapi juga precision, recall, dan F1-score.

In [ ]:
jumlah = data["label"].value_counts().reindex(KELAS)
persen = (jumlah / jumlah.sum() * 100).round(1)
display(pd.DataFrame({"jumlah": jumlah, "persen (%)": persen}))

fig, ax = plt.subplots(figsize=(6, 3.6))
batang = ax.bar(KELAS, jumlah, color=[WARNA[k] for k in KELAS])
ax.bar_label(batang, labels=[f"{j} ({p}%)" for j, p in zip(jumlah, persen)], padding=3)
ax.set_title("Jumlah pesan per kelas")
ax.set_ylabel("Jumlah pesan")
ax.set_ylim(0, jumlah.max() * 1.18)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FOLDER_HASIL / "distribusi_kelas.png", dpi=200)
plt.show()

### 4.2 Contoh pesan tiap kelas

In [ ]:
for kelas in KELAS:
    print(f"=== {kelas.upper()} ===")
    for teks in data[data["label"] == kelas]["teks"].sample(3, random_state=7):
        print("-", teks.replace("\n", " ")[:150])
    print()

### 4.3 Panjang pesan

Panjang pesan dihitung dalam jumlah kata.

In [ ]:
data["jumlah_kata"] = data["teks"].str.split().str.len()
display(data.groupby("label")["jumlah_kata"].describe().reindex(KELAS).round(1))

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.boxplot([data[data["label"] == k]["jumlah_kata"] for k in KELAS], patch_artist=True,
           boxprops={"facecolor": "#F1F2EC"}, medianprops={"color": "#1A1D17"})
ax.set_xticks(range(1, len(KELAS) + 1), KELAS)
ax.set_title("Panjang pesan per kelas")
ax.set_ylabel("Jumlah kata")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FOLDER_HASIL / "panjang_pesan.png", dpi=200)
plt.show()

### 4.4 Ciri khusus: tautan, nomor HP, nominal uang, dan kode layanan

Bagian ini mengukur berapa persen pesan di tiap kelas yang mengandung tautan, nomor HP, nominal uang, atau kode layanan operator (misalnya `*363#`). Hasilnya menjadi alasan mengapa ciri-ciri ini diganti dengan **token khusus** saat preprocessing.

In [ ]:
POLA_TAUTAN = re.compile(
    r"https?://\S+|www\.\S+|\b[a-z0-9-]+(?:\.[a-z0-9-]+)*\."
    r"(?:com|id|net|org|info|xyz|me|ly|co|biz|site|online|top|link|io)\b(?:/\S*)?"
)
POLA_NOMOR_HP = re.compile(r"(?:\+?62|\b0)8\d{7,12}\b")
POLA_UANG = re.compile(
    r"\brp\.?\s*\d[\d.,]*(?:\s*(?:rb|ribu|jt|juta|miliar))?\b|\b\d[\d.,]*\s*(?:rb|ribu|jt|juta|miliar)\b"
)
POLA_KODE_LAYANAN = re.compile(r"\*\d+(?:\*\d+)*#")

ciri = pd.DataFrame({
    "label": data["label"],
    "tautan": data["teks"].str.lower().str.contains(POLA_TAUTAN),
    "nomor HP": data["teks"].str.lower().str.contains(POLA_NOMOR_HP),
    "nominal uang": data["teks"].str.lower().str.contains(POLA_UANG),
    "kode layanan": data["teks"].str.contains(POLA_KODE_LAYANAN),
})
persen_ciri = (ciri.groupby("label").mean() * 100).reindex(KELAS).round(1)
display(persen_ciri)

ax = persen_ciri.T.plot.bar(figsize=(7, 3.8), color=[WARNA[k] for k in KELAS], rot=0)
ax.set_title("Persentase pesan yang mengandung ciri tertentu")
ax.set_ylabel("Persen pesan (%)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FOLDER_HASIL / "ciri_khusus.png", dpi=200)
plt.show()

## 5. Preprocessing teks

Langkah preprocessing dibuat **sederhana dan pasti**, karena langkah yang sama persis harus ditiru di aplikasi Android (Kotlin). Jika berbeda sedikit saja, vektor yang masuk ke Jaringan Syaraf Tiruan di HP tidak sama dengan saat pelatihan sehingga prediksinya salah.

| Langkah | Proses | Contoh |
|---|---|---|
| 1 | Case folding: semua huruf menjadi kecil | `Selamat ANDA` menjadi `selamat anda` |
| 2 | Ganti ciri khusus dengan token | tautan menjadi `tokentautan`, nomor HP menjadi `tokennomor`, nominal uang menjadi `tokenuang`, kode layanan menjadi `tokenkode`, angka lain menjadi `tokenangka` |
| 3 | Hapus tanda baca, simbol, dan emoji | `hadiah!!` menjadi `hadiah` |
| 4 | Tokenisasi: pecah kalimat menjadi kata | `klik link ini` menjadi `[klik, link, ini]` |
| 5 | Normalisasi kata gaul dengan kamus kecil | `yg` menjadi `yang`, `utk` menjadi `untuk` |
| 6 | Buang token satu huruf | sisa singkatan seperti `s`, `k` dari `S&K` |

**Yang sengaja tidak dilakukan:**
- **Stopword tidak dihapus**, karena kata seperti `anda` justru menjadi ciri pesan penipuan dan promo (dibuktikan di bagian 6.3).
- **Tanpa stemming**, karena library stemming bahasa Indonesia sulit ditiru sama persis di Kotlin.

**Alasan token khusus:** setiap tautan dan nomor HP berbeda-beda. Jika dibiarkan, setiap nomor menjadi kata baru yang hanya muncul sekali, kosakata membengkak, dan model tidak mengenali nomor atau tautan penipu yang baru. Dengan token, model belajar pola umum "pesan ini berisi tautan" yang tetap berlaku untuk penipu baru.

### 5.1 Kamus normalisasi kata gaul

Kamus disusun dari singkatan yang sering muncul di dataset. Kamus ini juga disimpan sebagai file JSON agar aplikasi Android memakai kamus yang sama.

In [ ]:
KAMUS_NORMALISASI = {
    "yg": "yang", "dgn": "dengan", "dg": "dengan", "utk": "untuk", "u": "untuk",
    "gak": "tidak", "ga": "tidak", "gk": "tidak", "nggak": "tidak", "ngga": "tidak",
    "enggak": "tidak", "tdk": "tidak", "sdh": "sudah", "udh": "sudah", "udah": "sudah",
    "blm": "belum", "bs": "bisa", "bsa": "bisa", "aja": "saja", "kalo": "kalau",
    "klo": "kalau", "sy": "saya", "aq": "aku", "ak": "aku", "km": "kamu",
    "kmu": "kamu", "dr": "dari", "krn": "karena", "jg": "juga", "lg": "lagi",
    "tp": "tapi", "pd": "pada", "sm": "sama", "trs": "terus", "bgt": "banget",
    "gmn": "bagaimana", "gimana": "bagaimana", "knp": "kenapa", "dpt": "dapat",
    "dapet": "dapat", "dptkan": "dapatkan", "ktk": "ketik", "hr": "hari", "bln": "bulan",
    "thn": "tahun", "jt": "juta", "rb": "ribu", "tlp": "telepon", "telp": "telepon",
    "hub": "hubungi", "plg": "pelanggan", "msh": "masih", "skrg": "sekarang",
    "kpd": "kepada", "ttg": "tentang", "spt": "seperti", "hrs": "harus", "org": "orang",
    "mhn": "mohon", "tsb": "tersebut", "sbg": "sebagai", "bsk": "besok", "mlm": "malam",
    "makasih": "terimakasih", "mksh": "terimakasih", "trims": "terimakasih",
}
print("Jumlah entri kamus:", len(KAMUS_NORMALISASI))

### 5.2 Fungsi preprocessing

Setiap langkah dibuat sebagai fungsi terpisah agar mudah dijelaskan dan ditiru satu per satu di Kotlin.

In [ ]:
def case_folding(teks):
    return teks.lower()


def ganti_token_khusus(teks):
    teks = POLA_TAUTAN.sub(" tokentautan ", teks)
    teks = POLA_KODE_LAYANAN.sub(" tokenkode ", teks)
    teks = POLA_NOMOR_HP.sub(" tokennomor ", teks)
    teks = POLA_UANG.sub(" tokenuang ", teks)
    teks = re.sub(r"\d+", " tokenangka ", teks)
    return teks


def hapus_simbol(teks):
    return re.sub(r"[^a-z\s]", " ", teks)


def tokenisasi(teks):
    return teks.split()


def normalisasi(token):
    return [KAMUS_NORMALISASI.get(kata, kata) for kata in token]


def buang_token_pendek(token):
    return [kata for kata in token if len(kata) > 1]


def preprocessing(teks):
    teks = case_folding(teks)
    teks = ganti_token_khusus(teks)
    teks = hapus_simbol(teks)
    token = tokenisasi(teks)
    token = normalisasi(token)
    return buang_token_pendek(token)

### 5.3 Contoh tahap demi tahap

Tabel ini bisa langsung dipakai di slide untuk menunjukkan proses preprocessing.

In [ ]:
contoh = data.groupby("label").sample(1, random_state=3)["teks"].tolist()
contoh.append("Selamat! No Anda menang undian Rp50.000.000 dr PT XYZ. Klik bit.ly/klaim-hadiah atau hub 081234567890 skrg")

langkah = []
for teks in contoh:
    t1 = case_folding(teks)
    t2 = ganti_token_khusus(t1)
    t3 = hapus_simbol(t2)
    t4 = tokenisasi(t3)
    t5 = buang_token_pendek(normalisasi(t4))
    langkah.append({
        "asli": teks,
        "1. case folding": t1,
        "2. token khusus": re.sub(r"\s+", " ", t2).strip(),
        "3. hapus simbol": re.sub(r"\s+", " ", t3).strip(),
        "4-6. token akhir": t5,
    })

for baris in langkah:
    for nama, nilai in baris.items():
        print(f"{nama:<18}: {nilai}")
    print("-" * 100)

### 5.4 Menerapkan preprocessing ke seluruh data

In [ ]:
data["token"] = data["teks"].apply(preprocessing)
data["teks_bersih"] = data["token"].str.join(" ")
data["jumlah_token"] = data["token"].str.len()

kosong = data[data["jumlah_token"] == 0]
print("Pesan yang menjadi kosong setelah preprocessing:", len(kosong))
display(kosong[["teks", "label"]])
data = data[data["jumlah_token"] > 0].reset_index(drop=True)

data[["teks", "teks_bersih", "label", "label_id"]].head(10)

## 6. Analisis kosakata

**Kosakata** adalah daftar seluruh kata unik. Ukurannya menentukan jumlah neuron di **input layer** Jaringan Syaraf Tiruan, karena setiap kata menjadi satu fitur masukan (materi MKBL pertemuan 5).

### 6.1 Pengaruh preprocessing terhadap ukuran kosakata

In [ ]:
def ukuran_kosakata(daftar_token):
    return len({kata for token in daftar_token for kata in token})


tanpa_token_khusus = data["teks"].apply(lambda t: buang_token_pendek(normalisasi(tokenisasi(hapus_simbol(case_folding(t))))))
hanya_lowercase = data["teks"].apply(lambda t: case_folding(t).split())

ringkasan_kosakata = pd.DataFrame({
    "tahap": ["Hanya case folding dan split", "Tanpa token khusus", "Preprocessing lengkap"],
    "ukuran kosakata": [ukuran_kosakata(hanya_lowercase), ukuran_kosakata(tanpa_token_khusus), ukuran_kosakata(data["token"])],
})
display(ringkasan_kosakata)

frekuensi = Counter(kata for token in data["token"] for kata in token)
print("Kata yang muncul hanya sekali:", sum(1 for jumlah in frekuensi.values() if jumlah == 1))

### 6.2 Kata yang paling sering muncul per kelas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
for ax, kelas in zip(axes, KELAS):
    hitung = Counter(kata for token in data[data["label"] == kelas]["token"] for kata in token)
    kata, jumlah_kata = zip(*hitung.most_common(15))
    ax.barh(kata[::-1], jumlah_kata[::-1], color=WARNA[kelas])
    ax.set_title(kelas)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("15 kata paling sering per kelas")
plt.tight_layout()
plt.savefig(FOLDER_HASIL / "kata_teratas.png", dpi=200)
plt.show()

### 6.3 Mengapa stopword tidak dihapus

Kata umum seperti `anda` biasanya dianggap stopword. Tabel di bawah menunjukkan persentase pesan di tiap kelas yang mengandung kata tersebut. Kata yang jauh lebih sering muncul di kelas penipuan atau promo justru berguna bagi model, sehingga tidak dihapus.

In [ ]:
kata_umum = ["anda", "kamu", "aku", "saya", "segera", "selamat", "yang", "di", "dan"]
tabel = pd.DataFrame({
    kata: data.groupby("label")["token"].apply(lambda s: (s.apply(lambda t: kata in t).mean() * 100)).reindex(KELAS)
    for kata in kata_umum
}).round(1)
tabel.index.name = "persen pesan (%)"
display(tabel)

## 7. Menyimpan hasil

| File | Isi | Dipakai untuk |
|---|---|---|
| `dataset_bersih.csv` | teks asli, label, label_id, teks hasil preprocessing | Pelatihan model (notebook berikutnya) |
| `kamus_normalisasi.json` | Kamus kata gaul | Disalin ke aplikasi Android agar preprocessing sama |
| `*.png` | Grafik eksplorasi data | Slide UTS |

Di Colab, semua file ada di folder `hasil` (panel **Files** di kiri) dan bisa diunduh sekaligus dalam bentuk zip.

In [ ]:
data[["teks", "label", "label_id", "teks_bersih", "sumber"]].to_csv(FOLDER_HASIL / "dataset_bersih.csv", index=False)
with open(FOLDER_HASIL / "kamus_normalisasi.json", "w", encoding="utf-8") as berkas:
    json.dump(KAMUS_NORMALISASI, berkas, ensure_ascii=False, indent=2, sort_keys=True)

for berkas in sorted(FOLDER_HASIL.iterdir()):
    print(f"{berkas.name:<26}{berkas.stat().st_size / 1024:>8.1f} KB")

In [ ]:
try:
    import shutil
    from google.colab import files

    shutil.make_archive("hasil_preprocessing", "zip", FOLDER_HASIL)
    files.download("hasil_preprocessing.zip")
except ImportError:
    print("Bukan di Google Colab, file tersimpan di folder:", FOLDER_HASIL.resolve())

## 8. Ringkasan untuk slide

In [ ]:
print("RINGKASAN DATA DAN PREPROCESSING")
print("-" * 40)
print(f"Total pesan          : {len(data)}")
for kelas in KELAS:
    n = (data["label"] == kelas).sum()
    print(f"  {kelas:<19}: {n} ({n / len(data) * 100:.1f}%)")
print(f"Data tambahan        : {(data['sumber'] == 'Data tambahan kelompok').sum()} pesan")
print(f"Rata-rata token/pesan: {data['jumlah_token'].mean():.1f}")
print(f"Ukuran kosakata      : {ukuran_kosakata(data['token'])} kata")
print(f"Entri kamus gaul     : {len(KAMUS_NORMALISASI)}")

## Langkah berikutnya

Notebook `02_pelatihan_model.ipynb` (setelah UTS):
1. Vektorisasi teks bersih dengan kosakata tetap (bag-of-words atau TF-IDF).
2. Membagi data latih dan data uji 80:20 secara stratified.
3. Melatih Multilayer Perceptron dengan algoritma Backpropagation (materi MKBL pertemuan 7).
4. Evaluasi: akurasi, precision, recall, F1-score, dan confusion matrix.
5. Konversi model ke TensorFlow Lite dan ekspor kosakata ke JSON.